# SmartEgg · Clasificador de visión (huevo / objeto / vacío)

Modelo de visión por computadora del Proyecto 2. Clasifica cada foto de la bandeja de la incubadora en una de tres clases:

| Clase | Qué hay en la foto | Acción del sistema |
|---|---|---|
| `huevo` | uno o más huevos, sin nada más | operación normal |
| `objeto` | cualquier cosa que no sea huevo: mano, pluma, llave, celular… | **alarma y bloqueo del sistema** |
| `vacio` | la bandeja sin nada | operación normal |

**Técnica:** *transfer learning* con **MobileNetV2** preentrenada en ImageNet. La red ya sabe reconocer bordes, texturas y formas; solo se le enseña a distinguir nuestras tres clases. Es una red liviana (≈2.2 M de parámetros con la cabeza nueva), así que corre en la CPU del servidor en la nube sin GPU.

**Dataset propio:** fotos reales tomadas con la cámara fija sobre la bandeja, con tres condiciones de luz (L1 foco encendido, L2 luz del cuarto, L3 poca luz), por tandas de 5 fotos.

La lógica está en `smartegg_vision.py`; este notebook la ejecuta paso a paso y muestra los resultados.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import smartegg_vision as sv
import torch

sv.set_seed()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET = sv.DEFAULT_DATASET      # Proyecto_2_SmartEgg/dataset/{huevo,objeto,vacio}
EPOCAS_CABEZA = 8
EPOCAS_AJUSTE = 12
print("Dispositivo:", DEVICE)
print("Dataset:", DATASET)

## 1. Dataset y separación por tandas

Las 5 fotos de una tanda son casi idénticas. Si se reparten al azar, algunas quedan en entrenamiento y sus "gemelas" en prueba, y el modelo parece mejor de lo que es. Por eso las fotos se **agrupan por tanda** (fotos tomadas con menos de 2 s de diferencia) y **cada tanda completa va a un solo conjunto**:

- **Entrenamiento (≈70 %)**: con estas aprende el modelo.
- **Validación (≈15 %)**: para elegir la mejor época y detener el entrenamiento.
- **Prueba (≈15 %)**: solo se usan al final, para medir el resultado real.

In [ ]:
samples = sv.list_samples(DATASET)
split = sv.split_by_group(samples)
groups = {s.group for s in samples}
print(f"{len(samples)} fotos en {len(groups)} tandas\n")
print(sv.describe_split(split))

In [ ]:
# Ejemplos de cada clase
rng = np.random.default_rng(0)
figure, axes = plt.subplots(len(sv.CLASSES), 4, figsize=(12, 3 * len(sv.CLASSES)))
for row, name in enumerate(sv.CLASSES):
    of_class = [s for s in samples if s.label == row]
    for axis, index in zip(axes[row], rng.choice(len(of_class), size=4, replace=len(of_class) < 4)):
        axis.imshow(sv.load_image(of_class[index].path).permute(1, 2, 0).numpy())
        axis.set_title(name)
        axis.axis("off")
figure.tight_layout()

## 2. Aumento de datos

En cada época, el modelo ve versiones ligeramente distintas de las fotos de entrenamiento. Así aprende a no depender de detalles que pueden cambiar en la incubadora:

- **Brillo, contraste y color** (±30 %): el foco se enciende y se apaga, y la luz del cuarto cambia.
- **Rotación ±6°, desplazamiento ±4 % y zoom ±5 %**: la bandeja se mueve un poco.
- **Espejo horizontal** y **desenfoque leve**.

**No se recorta la imagen**: un objeto en la orilla (escena 10 de `objeto`) debe seguir viéndose. Por la misma razón, la foto completa se estira a 224×224 en vez de recortar el centro.

In [ ]:
augment = sv.train_transform()
example = sv.load_image(split.train[0].path)
figure, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(example.permute(1, 2, 0).numpy())
axes[0].set_title("original")
mean = torch.tensor(sv.IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(sv.IMAGENET_STD).view(3, 1, 1)
for axis in axes[1:]:
    shown = (augment(example) * std + mean).clamp(0, 1)
    axis.imshow(shown.permute(1, 2, 0).numpy())
    axis.set_title("aumentada")
for axis in axes:
    axis.axis("off")
figure.tight_layout()

## 3. Modelo

MobileNetV2 con su clasificador de ImageNet (1000 clases) reemplazado por uno nuevo: `Dropout(0.3) → Linear(1280 → 3)`.

El entrenamiento tiene dos fases:

1. **Cabeza** (lr = 1e-3): la base queda congelada y solo aprende la capa nueva.
2. **Ajuste fino** (lr = 1e-4): se descongelan los últimos 5 bloques para adaptar las características a nuestras fotos.

Se usa `CrossEntropyLoss` con **pesos por clase** (por si las clases no quedan exactamente balanceadas) y *label smoothing* 0.05. En cada fase se guarda la época con **mejor F1 macro en validación**, y el entrenamiento se detiene si pasan 4 épocas sin mejora.

In [ ]:
model = sv.build_model(pretrained=True)
total = sum(p.numel() for p in model.parameters())
print(f"Parámetros: {total / 1e6:.2f} M")
train_loader, val_loader, test_loader = sv.make_loaders(split)
weights = sv.class_weights(split.train)
print("Pesos por clase:", {name: round(float(w), 3) for name, w in zip(sv.CLASSES, weights)})

In [ ]:
history = sv.train(model, train_loader, val_loader, weights,
                   head_epochs=EPOCAS_CABEZA, finetune_epochs=EPOCAS_AJUSTE, device=DEVICE)

In [ ]:
sv.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sv.plot_history(history, sv.RESULTS_DIR / "curvas_entrenamiento.png");

## 4. Evaluación en el conjunto de prueba

Fotos que el modelo **nunca vio** durante el entrenamiento ni la validación.

- **Precision**: de las fotos que el modelo llamó *X*, cuántas eran *X*.
- **Recall**: de las fotos que eran *X*, cuántas encontró.
- **F1**: media armónica de las dos anteriores. El **F1 macro** es el promedio de las tres clases.

La métrica más importante para la seguridad es el **recall de `objeto`**: un objeto no detectado no bloquea el sistema. Un falso positivo, en cambio, se filtra con la regla de 3 de 5 fotos del backend.

In [ ]:
labels, probabilities = sv.predict(model, test_loader, DEVICE)
metrics = sv.evaluate(labels, probabilities)
print(metrics["reporte"])
print(f"Exactitud: {metrics['exactitud']:.3f}")
print(f"F1 macro:  {metrics['f1_macro']:.3f}")
print(f"Recall de objeto: {metrics['por_clase']['objeto']['recall']:.3f}")

In [ ]:
sv.plot_confusion(metrics, sv.RESULTS_DIR / "matriz_confusion.png");

In [ ]:
# Fotos de prueba mal clasificadas
figure = sv.plot_errors(split.test, labels, probabilities, sv.RESULTS_DIR / "errores.png")
if figure is None:
    print("No hubo errores en el conjunto de prueba.")

### Análisis

*Completar después de entrenar con el dataset real:* qué clases se confunden, en qué condiciones de luz falla y por qué (ver la matriz de confusión y las fotos mal clasificadas).

## 5. Exportación a ONNX

El modelo se exporta a **ONNX** con la normalización y el `softmax` incluidos. El servicio de visión en Docker solo necesita **OpenCV** (`cv2.dnn`), sin PyTorch, y su imagen queda liviana.

- Entrada `imagen`: `1×3×224×224`, RGB, valores 0-255 (lo que entrega `cv2.dnn.blobFromImage(bgr, 1.0, (224, 224), swapRB=True)`).
- Salida `probabilidades`: `1×3`, en el orden de `clases.json`.

Al final se comprueba que OpenCV y PyTorch dan el mismo resultado.

In [ ]:
onnx_path = sv.export_onnx(model)
difference = sv.verify_onnx(model, onnx_path, split.test[:10])
sv.save_results(metrics, history, split)
print(f"Modelo: {onnx_path} ({onnx_path.stat().st_size / 1e6:.1f} MB)")
print(f"Diferencia máxima PyTorch vs OpenCV: {difference:.2e}")
print(f"Resultados en: {sv.RESULTS_DIR}")

## 6. Uso en el servicio de visión

```python
import cv2
net = cv2.dnn.readNetFromONNX("modelo/smartegg_vision.onnx")
blob = cv2.dnn.blobFromImage(foto_bgr, 1.0, (224, 224), swapRB=True, crop=False)
net.setInput(blob)
probabilidades = net.forward()[0]   # [huevo, objeto, vacio]
```

El servicio publica cada resultado en `smartegg/node-1/vision/result`. El backend bloquea el sistema (`system/lock`) cuando **3 de las últimas 5 fotos** salen como `objeto`, para no bloquear por un error aislado.

Para probar el modelo desde la terminal: `python ml/vision/predecir.py dataset/objeto`.